# UI Revision Evaluator — Qwen2.5-VL-32B

Evaluates before/after UI revision examples using Qwen2.5-VL-32B-Instruct.

**Before running:**
1. Enable internet access in the notebook's sidebar settings (off by default on Kaggle)
2. Select a GPU accelerator (T4 x2 required)

In [ ]:
# Clone the repo. Re-running this cell will fail if the directory already exists — that's fine.
!git clone https://github.com/VkumarStack/GenUI-Phase3

import os
os.chdir("/kaggle/working/GenUI-Phase3")

In [ ]:
# Install dependencies.
# qwen-vl-utils: Qwen's official helper for image tiling / processing
# bitsandbytes: enables 4-bit quantization to keep VRAM usage ~5GB on a T4
!pip install -q transformers accelerate bitsandbytes qwen-vl-utils

In [ ]:
import torch
from pathlib import Path

# Confirm GPU is available before attempting to load the model.
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  ({props.total_memory / 1e9:.1f} GB)")

In [ ]:
# Load Qwen2.5-VL-32B-Instruct in 4-bit quantization.
#
# This cell takes 5-10 minutes on first run — the model weights (~65GB on disk)
# are downloaded from HuggingFace and loaded into GPU memory.
# In 4-bit, active VRAM usage is roughly 16-18GB, spread across both T4s (32GB total).

from backends import HuggingFaceBackend

backend = HuggingFaceBackend(
    model="Qwen/Qwen2.5-VL-32B-Instruct",
    load_in_4bit=True,
)
print("Model loaded.")

In [ ]:
# Run evaluation over all examples in RevisionExamples/.

EXAMPLES_DIR = Path("RevisionExamples")

example_dirs = sorted(d for d in EXAMPLES_DIR.iterdir() if d.is_dir())

results = []
for example_dir in example_dirs:
    print(f"\n{'='*60}")
    print(f"Example: {example_dir.name}")
    print(f"Task:    {(example_dir / 'Task.txt').read_text().strip()}")
    print("-" * 60)

    result = backend.evaluate(example_dir)
    results.append(result)

    print(f"Verdict: {result['verdict']}")
    print(f"\nModel response:\n{result['response']}")